# Population-Level Statistics

This notebook inventories every persisted scalar `stats_*` column found in MALCA run products and plots a distribution for each one.

It focuses on scalar columns that survive flattening into parquet outputs. Table-like outputs from `compute_stats(...)` such as `nightly_table`, `by_camera`, or `seasons` are not written as ordinary `stats_*` columns and are therefore excluded from the histogram pass.


In [ ]:
import json
import re
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    pass

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 150,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'legend.frameon': True,
})


## Configuration

- Leave `RUN_DIR_NAMES` empty to auto-discover all run directories under `../../output/runs`.
- The notebook prefers the richest available candidate product per run, so vetted outputs still work as long as they retain `stats_*` columns.


In [ ]:
RUNS_ROOT = Path('../../output/runs')
RUN_DIR_NAMES = []

RESULT_PRIORITY = [
    'lc_events_vetted.parquet',
    'lc_events_spectra.parquet',
    'lc_events_neighbors.parquet',
    'lc_events_classified.parquet',
    'lc_events_enriched.parquet',
    'lc_events_characterized.parquet',
    'lc_events_filtered.parquet',
]

LOW_CARDINALITY_MAX = 12
PERCENTILE_CLIP = (0.01, 0.99)
BASE_COLORS = [
    '#1b9e77', '#d95f02', '#7570b3', '#e7298a',
    '#66a61e', '#e6ab02', '#a6761d', '#666666',
]

LOG10_COLUMNS = {
    'stats_file_points_total',
    'stats_file_points_kept_after_filter',
    'stats_time_span_days',
    'stats_n_unique_nights',
    'stats_cadence_mean_dt_days',
    'stats_cadence_median_dt_days',
    'stats_cadence_p05_dt_days',
    'stats_cadence_p95_dt_days',
    'stats_variability_reduced_chi2_vs_constant',
    'stats_variability_lomb_scargle_best_period_days',
    'stats_anderson_darling',
    'stats_gp_drw_tau',
    'stats_mhps_high',
    'stats_mhps_low',
    'stats_mhps_ratio',
}


In [ ]:
def infer_mag_bin_label(run_dir: Path) -> str:
    run_params = run_dir / 'run_params.json'
    if run_params.exists():
        try:
            params = json.loads(run_params.read_text())
            mag_bin = params.get('mag_bin')
            if mag_bin:
                return str(mag_bin)
            mag_min = params.get('mag_min')
            mag_max = params.get('mag_max')
            if mag_min is not None and mag_max is not None:
                return f'{float(mag_min):g}-{float(mag_max):g}'
        except Exception:
            pass

    match = re.search(r'(\d+(?:\.\d+)?)_(\d+(?:\.\d+)?)', run_dir.name)
    if match:
        return f'{float(match.group(1)):g}-{float(match.group(2)):g}'
    return run_dir.name


def pick_results_file(run_dir: Path) -> Path | None:
    results_dir = run_dir / 'results'
    if not results_dir.exists():
        return None

    for filename in RESULT_PRIORITY:
        exact = results_dir / filename
        if exact.exists():
            return exact

        stem = Path(filename).stem
        matches = sorted(
            results_dir.glob(f'{stem}_*.parquet'),
            key=lambda p: p.stat().st_mtime,
            reverse=True,
        )
        if matches:
            return matches[0]
    return None


def sort_labels(labels):
    def key(label):
        match = re.match(r'^(\d+(?:\.\d+)?)-(\d+(?:\.\d+)?)$', str(label))
        if match:
            return (0, float(match.group(1)), float(match.group(2)), str(label))
        return (1, str(label))

    return sorted(labels, key=key)


def discover_run_files() -> pd.DataFrame:
    if RUN_DIR_NAMES:
        run_dirs = [RUNS_ROOT / name for name in RUN_DIR_NAMES]
    else:
        run_dirs = sorted([p for p in RUNS_ROOT.iterdir() if p.is_dir()])

    rows = []
    for run_dir in run_dirs:
        result_file = pick_results_file(run_dir)
        if result_file is None:
            continue
        rows.append({
            'run_dir': run_dir,
            'run_name': run_dir.name,
            'mag_bin': infer_mag_bin_label(run_dir),
            'candidate_file': result_file,
        })

    if not rows:
        raise FileNotFoundError(f'No run products found under {RUNS_ROOT}')
    return pd.DataFrame(rows)


def load_population_frame(run_manifest: pd.DataFrame) -> pd.DataFrame:
    frames = []
    for row in run_manifest.itertuples(index=False):
        df = pd.read_parquet(row.candidate_file)
        df = df.copy()
        df['run_name'] = row.run_name
        df['mag_bin'] = row.mag_bin
        df['candidate_file'] = str(row.candidate_file)
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def numeric_series(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors='coerce')
    return s[np.isfinite(s)]


def is_binary_series(series: pd.Series) -> bool:
    s = numeric_series(series)
    if s.empty:
        return False
    return set(np.unique(s)) <= {0.0, 1.0}


def is_integer_like_series(series: pd.Series) -> bool:
    s = numeric_series(series)
    if s.empty:
        return False
    arr = s.to_numpy(dtype=float)
    return np.allclose(arr, np.round(arr))


def stats_group(col: str) -> str:
    if col.startswith('stats_file_points_') or col.startswith('stats_jd_') or col in {
        'stats_time_span_days',
        'stats_n_unique_nights',
        'stats_duty_cycle_fraction',
    }:
        return 'Coverage and timing'
    if col.startswith('stats_cadence_'):
        return 'Cadence'
    if col.startswith('stats_photometry_') or col.startswith('stats_clipped_') or col == 'stats_n_outliers_removed_robust_3sigma':
        return 'Photometry'
    if col.startswith('stats_error_and_snr_stats_'):
        return 'Errors and SNR'
    if col.startswith('stats_variability_lomb_scargle_') or col.startswith('stats_harmonics_') or col.startswith('stats_psi_'):
        return 'Period and harmonics'
    if col.startswith('stats_variability_') or col.startswith('stats_trend_'):
        return 'Core variability and trend'
    if col.startswith('stats_gp_drw_') or col == 'stats_iar_phi' or col.startswith('stats_mhps_'):
        return 'Stochastic models'
    return 'ALeRCE-style moments'


GROUP_ORDER = [
    'Coverage and timing',
    'Cadence',
    'Photometry',
    'Errors and SNR',
    'Core variability and trend',
    'ALeRCE-style moments',
    'Period and harmonics',
    'Stochastic models',
]


def plot_kind(series: pd.Series) -> str:
    s = numeric_series(series)
    if s.empty:
        return 'empty'
    if is_binary_series(s):
        return 'binary'
    if is_integer_like_series(s) and s.nunique() <= LOW_CARDINALITY_MAX:
        return 'discrete'
    return 'continuous'


def transform_series(col: str, series: pd.Series) -> tuple[pd.Series, str, str]:
    s = numeric_series(series)
    xlabel = col.removeprefix('stats_')
    transform = 'linear'

    if col.endswith('_fap'):
        s = s[s > 0]
        if not s.empty:
            s = -np.log10(s)
            xlabel = f'-log10({xlabel})'
            transform = '-log10'
        return s, xlabel, transform

    if col in LOG10_COLUMNS:
        s = s[s > 0]
        if not s.empty:
            s = np.log10(s)
            xlabel = f'log10({xlabel})'
            transform = 'log10'
    return s, xlabel, transform


def clip_series(series: pd.Series, kind: str) -> pd.Series:
    if series.empty or kind in {'binary', 'discrete'}:
        return series
    lo = series.quantile(PERCENTILE_CLIP[0])
    hi = series.quantile(PERCENTILE_CLIP[1])
    clipped = series[(series >= lo) & (series <= hi)]
    return clipped if not clipped.empty else series


def bins_for_series(series: pd.Series, kind: str):
    if kind == 'binary':
        return np.array([-0.5, 0.5, 1.5])
    if kind == 'discrete':
        lo = int(np.floor(series.min()))
        hi = int(np.ceil(series.max()))
        return np.arange(lo - 0.5, hi + 1.5, 1.0)
    return 40


def build_inventory(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    rows = []
    for col in cols:
        s = df[col]
        numeric = numeric_series(s)
        _, _, transform = transform_series(col, s)
        rows.append({
            'column': col,
            'group': stats_group(col),
            'dtype': str(s.dtype),
            'non_null': int(s.notna().sum()),
            'completeness': float(s.notna().mean()),
            'n_unique_non_null': int(s.dropna().nunique()),
            'plot_kind': plot_kind(s),
            'transform': transform,
            'min': float(numeric.min()) if not numeric.empty else np.nan,
            'max': float(numeric.max()) if not numeric.empty else np.nan,
        })
    inventory = pd.DataFrame(
        rows,
        columns=[
            'column', 'group', 'dtype', 'non_null', 'completeness',
            'n_unique_non_null', 'plot_kind', 'transform', 'min', 'max',
        ],
    )
    if inventory.empty:
        return inventory
    return inventory.sort_values(['group', 'column']).reset_index(drop=True)


def summarize_column(df: pd.DataFrame, column: str) -> pd.DataFrame:
    rows = []
    labels = sort_labels(df['mag_bin'].dropna().unique()) + ['ALL']
    for label in labels:
        subset = df[column] if label == 'ALL' else df.loc[df['mag_bin'] == label, column]
        s = numeric_series(subset)
        if s.empty:
            continue
        rows.append({
            'column': column,
            'mag_bin': label,
            'count': int(s.size),
            'mean': float(s.mean()),
            'std': float(s.std(ddof=1)) if s.size > 1 else np.nan,
            'median': float(s.median()),
            'mad': float(np.median(np.abs(s - s.median()))),
            'p05': float(s.quantile(0.05)),
            'p25': float(s.quantile(0.25)),
            'p75': float(s.quantile(0.75)),
            'p95': float(s.quantile(0.95)),
        })
    return pd.DataFrame(rows)


def palette_for(labels: list[str]) -> dict[str, str]:
    ordered = sort_labels(labels)
    return {label: BASE_COLORS[i % len(BASE_COLORS)] for i, label in enumerate(ordered)}


def plot_group_histograms(df: pd.DataFrame, cols: list[str], title: str, split_col: str = 'mag_bin') -> None:
    if not cols:
        return

    labels = sort_labels(df[split_col].dropna().unique())
    palette = palette_for(labels)
    ncols = min(3, len(cols))
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 3.8 * nrows), squeeze=False)
    fig.suptitle(title, fontsize=15, fontweight='bold', y=1.02)

    for idx, col in enumerate(cols):
        ax = axes[idx // ncols, idx % ncols]
        kind = INVENTORY_MAP[col]['plot_kind']
        xlabel = col.removeprefix('stats_')
        transform_label = INVENTORY_MAP[col]['transform']
        if transform_label == 'log10':
            xlabel = f'log10({xlabel})'
        elif transform_label == '-log10':
            xlabel = f'-log10({xlabel})'

        for label in labels:
            series, _, _ = transform_series(col, df.loc[df[split_col] == label, col])
            series = clip_series(series, kind)
            if series.empty:
                continue
            bins = bins_for_series(series, kind)
            ax.hist(
                series,
                bins=bins,
                density=True,
                histtype='step',
                linewidth=1.8,
                color=palette[label],
                label=label,
            )

        ax.set_title(col.removeprefix('stats_'), fontsize=10)
        ax.set_xlabel(xlabel, fontsize=9)
        ax.set_ylabel('density', fontsize=9)
        ax.tick_params(labelsize=8)
        if idx == 0:
            ax.legend(fontsize=8, title=split_col)

    for idx in range(len(cols), nrows * ncols):
        axes[idx // ncols, idx % ncols].set_visible(False)

    fig.tight_layout()
    plt.show()


In [ ]:
run_manifest = discover_run_files()
display(run_manifest[['run_name', 'mag_bin', 'candidate_file']])

df_all = load_population_frame(run_manifest)
print(f'Loaded {len(df_all):,} rows from {run_manifest.shape[0]} run product(s).')


In [ ]:
stats_cols_all = sorted([c for c in df_all.columns if c.startswith('stats_')])
scalar_stats_cols = [
    c for c in stats_cols_all
    if pd.api.types.is_numeric_dtype(df_all[c]) and df_all[c].notna().any()
]
skipped_stats_cols = sorted(set(stats_cols_all) - set(scalar_stats_cols))

inventory = build_inventory(df_all, scalar_stats_cols)
INVENTORY_MAP = inventory.set_index('column').to_dict(orient='index')

print(f'{len(stats_cols_all)} stats_* columns found')
print(f'{len(scalar_stats_cols)} numeric scalar stats will be histogrammed')
if skipped_stats_cols:
    print(f'{len(skipped_stats_cols)} stats_* columns skipped because they are non-numeric or all-null')

display(inventory)


## Coverage Audit

The table below makes the scope explicit: every discovered scalar numeric `stats_*` column is assigned to a plot family and a plotting mode.


In [ ]:
coverage_audit = (
    inventory.groupby(['group', 'plot_kind'])['column']
    .count()
    .rename('n_columns')
    .reset_index()
    .sort_values(['group', 'plot_kind'])
)
display(coverage_audit)


## Long-Form Summary Table

`summary_long` keeps per-column summary statistics for each magnitude bin plus an `ALL` aggregate. Query it by column name when needed.


In [ ]:
summary_frames = [summarize_column(df_all, col) for col in scalar_stats_cols]
summary_long = (
    pd.concat(summary_frames, ignore_index=True)
    if summary_frames
    else pd.DataFrame(columns=['column', 'mag_bin', 'count', 'mean', 'std', 'median', 'mad', 'p05', 'p25', 'p75', 'p95'])
)
display(summary_long.head(20))


## Histograms For Every Persisted Scalar Stat

Each section below plots every discovered scalar `stats_*` column in that family. Continuous fields use percentile clipping to keep extreme outliers from flattening the view; binary and low-cardinality discrete fields keep their native bins.


In [ ]:
for group_name in GROUP_ORDER:
    cols = [c for c in scalar_stats_cols if INVENTORY_MAP[c]['group'] == group_name]
    if cols:
        plot_group_histograms(df_all, cols, title=group_name)


## Notes

- Missing Lomb-Scargle or harmonic columns usually mean those stats were not computed for the run product.
- Table-like outputs from `compute_stats(...)` are intentionally not part of the histogram inventory because they are not persisted as simple `stats_*` scalars.
- If you only want a subset of runs, set `RUN_DIR_NAMES` at the top and rerun the notebook.
